# SANAD Adaptive v0.1 — real ESPARGOS CSI pilot

**Research-only** variable-array/antenna prototype; **NOT frozen RC3**. This notebook uses genuine CSI recordings (static and walking) and then mathematically injects controlled faults. Outputs automatically save to **Google Drive**. No real antenna actuation; no certified field-fault ground truth.

Data source: https://espargos.net/datasets/data/espargos-0001/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, zipfile, os, json, datetime
BASE = Path('/content/drive/MyDrive/SANAD_Adaptive/Prototype_v0')
SOURCE_ZIP = BASE/'SANAD_Adaptive_v0.zip'
assert SOURCE_ZIP.exists(), f'Archive missing in Drive: {SOURCE_ZIP}'
WORK = Path('/content/SANAD_Adaptive_v0')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    bad=[f for f in z.namelist() if f.startswith('/') or '..' in Path(f).parts]
    assert not bad, bad
    z.extractall(WORK)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK/'requirements-espargos.txt')])
print('Experimental SANAD Adaptive loaded from Drive archive:', SOURCE_ZIP)

In [ ]:
# Optional validation of the math engine on small synthetic fixtures.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytest>=8,<10'])
env = dict(os.environ, PYTHONPATH=str(WORK))
subprocess.check_call([sys.executable, '-m', 'pytest', '-q', str(WORK/'tests')], env=env)

In [ ]:
# Download only the TWO small ESPARGOS recordings, run the actual CSI pilot,
# and SAVE JSON, CSV and figure directly into Google Drive, automatically.
RUNSTAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = Path('/content/drive/MyDrive/SANAD_Adaptive/Results/ESPARGOS')/RUNSTAMP
OUT.mkdir(parents=True, exist_ok=False)
DATA = Path('/content/SANAD_Adaptive_Datasets')
DATA.mkdir(exist_ok=True)
command=[sys.executable, str(WORK/'experiments/run_espargos_adaptive.py'),
         '--download','--data-dir',str(DATA),'--output',str(OUT),'--window','16']
print('Output Drive location:',OUT)
subprocess.check_call(command, env=env)
print('RESULTS SAVED TO DRIVE:',OUT)

In [ ]:
# Show outputs and write an integrity file next to the results.
import hashlib
from IPython.display import display, Image
result=json.loads((OUT/'adaptive_espargos_report.json').read_text())
print(json.dumps({k:result[k] for k in ('status','static','walking','injections','limitations')},indent=2))
if (OUT/'adaptive_espargos_scores.png').exists():
    display(Image(filename=str(OUT/'adaptive_espargos_scores.png')))

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(1048576),b''):h.update(part)
    return h.hexdigest()
manifest={f.name:{'sha256':sha(f),'size_bytes':f.stat().st_size} for f in OUT.iterdir() if f.is_file()}
(OUT/'RESULTS_SHA256.json').write_text(json.dumps(manifest,indent=2))
print('Saved SHA-256 manifest; files:',sorted(p.name for p in OUT.iterdir()))